In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False
print('환경 설정 완료!')

환경 설정 완료!


In [4]:
# [예제 1] 결측값 확인 (code7_1, code7_2)
import pandas as pd
import numpy as np

df = pd.read_csv('data/iris/iris.csv')

# 인위적으로 결측값 주입
df.iloc[0, 1] = pd.NA
df.iloc[0, 2] = pd.NA
df.iloc[1, 2] = np.nan
df.iloc[2, 3] = None

print('컬럼별 결측값 수:')
print(df.isnull().sum())
print()
print('결측값이 있는 행:')
print(df.loc[df.isnull().sum(axis=1) > 0, :])

컬럼별 결측값 수:
sepal_length    0
sepal_width     1
petal_length    2
petal_width     1
species         0
dtype: int64

결측값이 있는 행:
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          NaN           NaN          0.2  setosa
1           4.9          3.0           NaN          0.2  setosa
2           4.7          3.2           1.3          NaN  setosa


In [5]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,NaN,NaN,0.2,setosa
1,4.9,3.0,NaN,0.2,setosa
2,4.7,3.2,1.3,NaN,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [6]:
# [에제2] 데이터 정제 - 로우 삭제 및 인덱스 재설정
print('정제 전 행 수:', len(df))
df_clean = df.dropna().reset_index(drop=True)  # 결측값 행 제거 + 인덱스 재설정
print('정제 후 행 수:', len(df_clean))
print(df_clean.head())

정제 전 행 수: 150
정제 후 행 수: 147
   sepal_length  sepal_width  petal_length  petal_width species
0           4.6          3.1           1.5          0.2  setosa
1           5.0          3.6           1.4          0.2  setosa
2           5.4          3.9           1.7          0.4  setosa
3           4.6          3.4           1.4          0.3  setosa
4           5.0          3.4           1.5          0.2  setosa


In [12]:
# [예제 3] 이상치 탐지 (code7_4)
from scipy import stats

sw = df_clean['sepal_width']

# Z-score 방법 (|z| > 2 기준)
z = np.abs(stats.zscore(sw))            # 결측치(NaN) 존재 시 이상해짐
outliers_z = sw[z > 2]
print('Z-score 이상치 (|z|>2):')
print(outliers_z.values)

# IQR 방법
Q1 = sw.quantile(0.25)
Q3 = sw.quantile(0.75)
IQR = Q3 - Q1
outliers_iqr = sw[(sw < Q1 - IQR*1.5) | (sw > Q3 + IQR*1.5)]
print(f'\nIQR 이상치 (Q1-1.5×IQR={Q1-1.5*IQR:.2f}, Q3+1.5×IQR={Q3+1.5*IQR:.2f}):')
print(outliers_iqr.values)

# 이상치 제거
clean = sw.loc[~sw.isin(outliers_iqr)]
print(f'\n원본 크기: {len(sw)} → 이상치 제거 후: {len(clean)}')


Z-score 이상치 (|z|>2):
[4.  4.4 4.1 4.2 2. ]

IQR 이상치 (Q1-1.5×IQR=2.05, Q3+1.5×IQR=4.05):
[4.4 4.1 4.2 2. ]

원본 크기: 147 → 이상치 제거 후: 143


In [18]:
# [예제 4] 정렬 (sort_values), 순위 (rank) (code7_5, code7_6)

# 정렬
print('Sepal_Length 오름차순 상위 5행:')
print(df_clean.sort_values('sepal_length').head())

print('\nSpecies, Sepal_Width 복합 정렬 상위 5행:')
print(df_clean.sort_values(['species', 'sepal_width']).head())

# 순위
print('\nPetal_Length 오름차순 순위 상위 5:')
print(df_clean['petal_length'].rank().astype(int).head())

Sepal_Length 오름차순 상위 5행:
    sepal_length  sepal_width  petal_length  petal_width species
10           4.3          3.0           1.1          0.1  setosa
39           4.4          3.2           1.3          0.2  setosa
35           4.4          3.0           1.3          0.2  setosa
5            4.4          2.9           1.4          0.2  setosa
38           4.5          2.3           1.3          0.3  setosa

Species, Sepal_Width 복합 정렬 상위 5행:
    sepal_length  sepal_width  petal_length  petal_width species
38           4.5          2.3           1.3          0.3  setosa
5            4.4          2.9           1.4          0.2  setosa
9            4.8          3.0           1.4          0.1  setosa
10           4.3          3.0           1.1          0.1  setosa
22           5.0          3.0           1.6          0.2  setosa

Petal_Length 오름차순 순위 상위 5:
0    27
1    15
2    43
3    15
4    27
Name: petal_length, dtype: int64


### groupby 집계

In [19]:
# [예제6] groupby
df = pd.read_csv('data/iris/iris.csv')
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [23]:
# 단일 그룹 집계
print('품종별 평균 : ')
display(df.groupby('species').mean().round(2))

품종별 평균 : 


,sepal_length,sepal_width,petal_length,petal_width
species,,,,
setosa,5.01,3.42,1.46,0.24
versicolor,5.94,2.77,4.26,1.33
virginica,6.59,2.97,5.55,2.03


In [28]:
# 다중 그룹 집계
mpg_df = pd.read_csv('data/auto-mpg.csv')
mpg_df.head()

mpg_df.groupby(['cylinders', 'weight']).max().head(3)

mpg  displacement horsepower  acceleration  model year  \
cylinders weight                                                            
3         2124    18.0          70.0         90          13.5          73   
          2330    19.0          70.0         97          13.5          72   
          2420    23.7          70.0        100          12.5          80   

                  origin         car name  
cylinders weight                           
3         2124         3        maxda rx3  
          2330         3  mazda rx2 coupe  
          2420         3    mazda rx-7 gs

### concat & merge

In [30]:
# [예제 7] concat으로 DataFrame 결합 (code7_10)
df1 = pd.DataFrame([[169, 58, 1.0], [172, 73, 1.2], [184, 82, 0.7]],
                   columns=['height', 'weight', 'eye'])
df2 = pd.DataFrame([[176, 71, 0.8, 'M'], [169, 62, 0.7, 'F']],
                   columns=['height', 'weight', 'eye', 'gender'])
print(df1.head(3), df2.head(3), sep = '\n')

   height  weight  eye
0     169      58  1.0
1     172      73  1.2
2     184      82  0.7
   height  weight  eye gender
0     176      71  0.8      M
1     169      62  0.7      F


In [33]:
# 행방향 결합
df12 = pd.concat([df1, df2]).reset_index(drop = True)
df12

,height,weight,eye,gender
0,169,58,1.0,NaN
1,172,73,1.2,NaN
2,184,82,0.7,NaN
3,176,71,0.8,M
4,169,62,0.7,F


In [35]:
# [예제8] merge로 JOIN

df1 = pd.DataFrame([['a', 90], ['b', 80], ['c', 40]], columns=['name', 'kor'])
df2 = pd.DataFrame([['a', 75], ['b', 60], ['d', 90]], columns=['name', 'math'])

print(df1, '\n', df2)

  name  kor
0    a   90
1    b   80
2    c   40 
   name  math
0    a    75
1    b    60
2    d    90


In [37]:
# inner join
# 이름이 같은 행만 join
df1.merge(df2, on = 'name')

,name,kor,math
0,a,90,75
1,b,80,60


In [39]:
# left outer join : df1 기준
df1.merge(df2, how = 'left', on = 'name')      # left outer join(df1) / join key : 'name'

,name,kor,math
0,a,90,75.0
1,b,80,60.0
2,c,40,NaN


In [43]:
# outer join : 전체 포함
# df1.merge(df2, how = 'outer', on = 'name')      # full outer join
pd.merge(df1, df2, how = 'outer', on = 'name')

,name,kor,math
0,a,90.0,75.0
1,b,80.0,60.0
2,c,40.0,NaN
3,d,NaN,90.0
